# Part 3 — Spark Structured Streaming consumer

Reads the `financial_data` Kafka topic in real time, applies a typed schema to the JSON payload, then demonstrates three transformations required by the assignment:

1. **Filter abnormal values** — drop ticks that are clearly out of range (non-positive price, price outside a plausible band per symbol).
2. **Moving average price** — 1-minute sliding window over the event-time stream, updated every 10 s.
3. **Aggregate per symbol** — live min / max / avg / count per crypto symbol since start of day.

Everything runs in **local[*]** mode for the same reason as Part 2 (Python 3.11 driver vs 3.8 workers).

**Prerequisite.** Start `02_producer_binance.ipynb` in another notebook window first, so ticks keep arriving in the topic while this consumer is running.

## 1. Build the Spark session

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FinancialStreamProcessor")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready")

## 2. Declare the JSON schema

Structured Streaming does **not** infer schema from Kafka — we must provide one. Our producer emits `{symbol, price, volume, timestamp}` and we enforce the exact same types on the consumer side. Any tick whose `value` column does not parse will be dropped by `from_json`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

SCHEMA = StructType([
    StructField("symbol",    StringType(), False),
    StructField("price",     DoubleType(), False),
    StructField("volume",    DoubleType(), False),
    StructField("timestamp", LongType(),   False),
])
SCHEMA

## 3. Read the Kafka topic as a streaming DataFrame

`startingOffsets="latest"` means we begin from the live end of the topic (we don't replay history). Change to `"earliest"` to re-process everything.

In [ ]:
from pyspark.sql.functions import col, from_json, from_unixtime

raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "financial_data")
    .option("startingOffsets", "latest")
    .load()
)

parsed = (
    raw.selectExpr("CAST(value AS STRING) AS json_str",
                   "timestamp AS kafka_ts")
       .select(from_json(col("json_str"), SCHEMA).alias("d"), "kafka_ts")
       .select("d.symbol", "d.price", "d.volume", "d.timestamp", "kafka_ts")
       # Convert the producer's epoch-ms into a proper TimestampType event-time column
       .withColumn("event_time", from_unixtime(col("timestamp") / 1000).cast("timestamp"))
)
parsed.printSchema()

## 4. Transformation 1 — Filter abnormal values

Two guardrails:
- reject non-positive prices or volumes (schema-level sanity),
- reject prices outside a plausible band per symbol (business-level sanity — catches producer bugs or market glitches).

The bands are loose (±50 % around a rough price anchor) so normal market volatility is never filtered out.

In [ ]:
from pyspark.sql.functions import lit, when

# Rough mid-price anchor per symbol. Used only for outlier detection.
ANCHOR = {
    "BTCUSDT": 77000.0,
    "ETHUSDT": 2300.0,
    "SOLUSDT":  180.0,
    "BNBUSDT":  630.0,
    "ADAUSDT":    0.55,
}

# Build a broadcast-friendly expression: anchor per symbol (null for unknown).
anchor_expr = None
for sym, px in ANCHOR.items():
    branch = when(col("symbol") == sym, lit(px))
    anchor_expr = branch if anchor_expr is None else anchor_expr.when(col("symbol") == sym, lit(px))
anchor_expr = anchor_expr.otherwise(lit(None))

with_anchor = parsed.withColumn("anchor_px", anchor_expr)

clean = (
    with_anchor
    .filter(col("price") > 0)
    .filter(col("volume") >= 0)
    # within 50% of the anchor when the symbol is known (unknown symbols pass through)
    .filter(
        col("anchor_px").isNull() |
        ((col("price") >= col("anchor_px") * 0.5) & (col("price") <= col("anchor_px") * 1.5))
    )
    .drop("anchor_px")
)
clean.printSchema()

In [ ]:
from pyspark.sql.functions import lit, when

# Rough mid-price anchor per symbol. Used only for outlier detection.
# Picked generously (±70% band below, ×3 band above) so normal market
# volatility is never filtered out. Only extreme outliers get dropped.
ANCHOR = {
    "BTCUSDT": 77000.0,
    "ETHUSDT":  2300.0,
    "SOLUSDT":    85.0,
    "BNBUSDT":   630.0,
    "ADAUSDT":     0.25,
}
LOWER_RATIO = 0.3   # price must be >= 0.3 x anchor
UPPER_RATIO = 3.0   # price must be <= 3.0 x anchor

# Build a case-when expression mapping each symbol to its anchor.
anchor_expr = None
for sym, px in ANCHOR.items():
    branch = when(col("symbol") == sym, lit(px))
    anchor_expr = branch if anchor_expr is None else anchor_expr.when(col("symbol") == sym, lit(px))
anchor_expr = anchor_expr.otherwise(lit(None))

with_anchor = parsed.withColumn("anchor_px", anchor_expr)

clean = (
    with_anchor
    .filter(col("price") > 0)
    .filter(col("volume") >= 0)
    # Symbols with a known anchor must stay within [0.3 x, 3.0 x] of it.
    # Unknown symbols pass through (we don't judge what we don't know).
    .filter(
        col("anchor_px").isNull() |
        ((col("price") >= col("anchor_px") * LOWER_RATIO) &
         (col("price") <= col("anchor_px") * UPPER_RATIO))
    )
    .drop("anchor_px")
)
clean.printSchema()

In [ ]:
from pyspark.sql.functions import window, avg, count

moving_avg = (
    clean
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "1 minute", "10 seconds"),
        col("symbol"),
    )
    .agg(
        avg("price").alias("avg_price"),
        count("*").alias("n_ticks"),
    )
    .select(
        col("symbol"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("avg_price"),
        col("n_ticks"),
    )
)

## 6. Transformation 3 — Per-symbol aggregation (running)

A **stateless running aggregation** (no window) — since stream start, per symbol: min / max / avg price and running volume. This answers *“aggregate per symbol”* directly.

In [ ]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

per_symbol = (
    clean.groupBy("symbol")
    .agg(
        _min("price").alias("min_price"),
        _max("price").alias("max_price"),
        avg("price").alias("avg_price"),
        _sum("volume").alias("total_volume"),
        count("*").alias("n_ticks"),
    )
)

## 7. Start the streaming queries

We launch three streaming queries writing to the **console** sink so you can see the live output directly under the cells. In Part 4 these queries will instead write to HDFS (Parquet, partitioned).

A `checkpointLocation` is provided for each query — Spark persists offsets + state there so that, if the job crashes and is restarted, it resumes from where it stopped (exactly-once state, at-least-once delivery).

In [ ]:
CKPT_ROOT = "/tmp/lab3_checkpoints"

q_clean = (
    clean.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 5)
    .option("checkpointLocation", f"{CKPT_ROOT}/clean")
    .trigger(processingTime="10 seconds")
    .queryName("clean_console")
    .start()
)

q_window = (
    moving_avg.writeStream
    .format("console")
    .outputMode("update")          # 'update' = only windows whose aggregates changed
    .option("truncate", False)
    .option("checkpointLocation", f"{CKPT_ROOT}/moving_avg")
    .trigger(processingTime="15 seconds")
    .queryName("moving_avg_console")
    .start()
)

q_per_symbol = (
    per_symbol.writeStream
    .format("console")
    .outputMode("complete")        # 'complete' = full running table every trigger
    .option("truncate", False)
    .option("checkpointLocation", f"{CKPT_ROOT}/per_symbol")
    .trigger(processingTime="15 seconds")
    .queryName("per_symbol_console")
    .start()
)

for q in (q_clean, q_window, q_per_symbol):
    print(q.name, "->", q.id, "isActive=", q.isActive)

## 8. Monitor + stop

`awaitTermination()` blocks the cell forever (one common pattern). If you just want to peek and keep going, use the query handles directly:

In [ ]:
# Peek at progress without blocking.
for q in (q_clean, q_window, q_per_symbol):
    p = q.lastProgress
    if p:
        print(f"{q.name}: batchId={p['batchId']} "
              f"inputRowsPerSecond={p['inputRowsPerSecond']:.2f} "
              f"processedRowsPerSecond={p['processedRowsPerSecond']:.2f}")

In [ ]:
# Run this cell to stop all three queries cleanly.
for q in (q_clean, q_window, q_per_symbol):
    q.stop()
print("all queries stopped")

In [ ]:
spark.stop()